# Gerador de Fato: Vendas

**Objetivo:** simular a geração diária de transações de vendas (grão: item vendido por transação), distribuídas entre os centros de distribuição de um país, respeitando pesos de distribuição e a sazonalidade de Natal (+30% em 24/12).

**Dependências:**
- Lê os arquivos JSON já gerados na Landing Zone para `dim_produtos`, `dim_lojas` e `dim_representantes` (não redefine essas dimensões, evitando duplicação/inconsistência de dados mestres)

**Widgets:**
- `pais` (dropdown: brasil / argentina / mexico) — define para qual país a carga de vendas será gerada nesta execução, permitindo que o Workflow chame este mesmo notebook 3 vezes (uma por país)
- `data_venda` (texto, padrão = data de hoje) — permite simular qualquer data, incluindo 24/12 para testar a sazonalidade

**Volume diário (por país):**
| País | Linhas/dia |
|---|---|
| Brasil | 10.000 |
| México | 3.000 |
| Argentina | 500 |

**Saída:** arquivo(s) JSON gravado(s) na Landing Zone (`/Volumes/poc_latam_food/landing/blob_simulado/vendas/pais=<pais>/data=<data_venda>/`).

**Observação de arquitetura:** este notebook grava as vendas **apenas em moeda local** (`valor_unitario_moeda_local`, `valor_total_moeda_local`, `moeda`). A conversão para USD (`cambio_usado`, `valor_total_usd`) é responsabilidade da camada Silver, consultando a API de câmbio — mantendo a Raw fiel ao que um sistema de origem real produziria (sem conversão de moeda embutida na transação).

In [0]:
# Instalações
%pip install holidays

In [0]:
# Reinicialização do Python
dbutils.library.restartPython()

In [0]:
# Instalações e configurações iniciais

import sys
import uuid
import random
import holidays
from datetime import date, datetime
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

sys.path.append("/Workspace/Users/bruno.quelestech@outlook.com/poc-lakehouse-food-latam/src")

print("Path configurado com sucesso.")

In [0]:
# Widget

dbutils.widgets.dropdown("pais", "brasil", ["brasil", "argentina", "mexico"])
dbutils.widgets.text("data_venda", "")

pais_selecionado = dbutils.widgets.get("pais")

# Se o widget 'data_venda' estiver vazio, usa a data de hoje como padrão
from datetime import date

data_venda_texto = dbutils.widgets.get("data_venda")
data_venda = date.fromisoformat(data_venda_texto) if data_venda_texto else date.today()

print(f"País selecionado: {pais_selecionado}")
print(f"Data da venda: {data_venda}")

In [0]:
# Identifica feriado no pais

mapa_holidays = {
    "brasil": holidays.Brazil(),
    "argentina": holidays.Argentina(),
    "mexico": holidays.Mexico(),
}

calendario_feriados = mapa_holidays[pais_selecionado]

eh_fim_de_semana = data_venda.weekday() >= 5
eh_feriado = data_venda in calendario_feriados

eh_dia_util = not eh_fim_de_semana and not eh_feriado

print(f"Data: {data_venda} ({data_venda.strftime('%A')})")
print(f"É fim de semana? {eh_fim_de_semana}")
print(f"É feriado em {pais_selecionado}? {eh_feriado}")
print(f"É dia útil? {eh_dia_util}")

In [0]:
# Log de controle (feriado/dia util)

caminho_log_execucao = "/Volumes/poc_latam_food/landing/blob_simulado/logs/execucao_vendas"

motivo_bloqueio = None
if eh_fim_de_semana:
    motivo_bloqueio = "fim_de_semana"
elif eh_feriado:
    motivo_bloqueio = "feriado"

log_execucao = [(
    data_venda.isoformat(),
    pais_selecionado,
    eh_dia_util,
    motivo_bloqueio,
    datetime.now().isoformat(),
)]

schema_log = StructType([
    StructField("data_venda", StringType(), True),
    StructField("pais", StringType(), True),
    StructField("eh_dia_util", BooleanType(), True),
    StructField("motivo_bloqueio", StringType(), True),
    StructField("data_execucao", StringType(), True),
])

df_log = spark.createDataFrame(log_execucao, schema=schema_log)

(
    df_log
    .write
    .mode("append")
    .json(caminho_log_execucao)
)

print(f"Log de execução registrado: dia útil = {eh_dia_util}, motivo = {motivo_bloqueio}")

In [0]:
# Condição para processar vendas no dia

if not eh_dia_util:
    print(f"Execução interrompida: {data_venda} não é dia útil para {pais_selecionado} "
          f"(motivo: {motivo_bloqueio}). Nenhuma venda será gerada.")
    dbutils.notebook.exit("Bloqueado: dia não útil")

In [0]:
# Ler as 3 dimensões da Landing Zone

caminho_produtos = "/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/produtos"
caminho_lojas = "/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/lojas"
caminho_representantes = "/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/representantes"

df_produtos = spark.read.json(caminho_produtos)

mapa_pais = {"brasil": "Brasil", "argentina": "Argentina", "mexico": "Mexico"}
pais_filtro = mapa_pais[pais_selecionado]

# Filtro case-insensitive, usando lower() nos dois lados da comparação
df_lojas = spark.read.json(caminho_lojas).filter(f"lower(pais) = lower('{pais_filtro}')")
df_representantes = spark.read.json(caminho_representantes).filter(f"lower(pais) = lower('{pais_filtro}')")

print(f"Produtos disponíveis: {df_produtos.count()}")
print(f"Lojas do país '{pais_filtro}': {df_lojas.count()}")
print(f"Representantes do país '{pais_filtro}': {df_representantes.count()}")

df_representantes.display()

In [0]:
# Simula vendas 

volume_base_por_pais = {
    "brasil": 10000,
    "mexico": 3000,
    "argentina": 500,
}

volume_do_dia = volume_base_por_pais[pais_selecionado]

# Sazonalidade: Natal (24/12) aumenta o volume em 30%
eh_natal = (data_venda.month == 12) and (data_venda.day == 24)

if eh_natal:
    volume_do_dia = int(volume_do_dia * 1.30)

print(f"Volume base: {volume_base_por_pais[pais_selecionado]}")
print(f"É véspera de Natal? {eh_natal}")
print(f"Volume final do dia: {volume_do_dia}")

In [0]:
# Trazendo os pesos-base de cada loja para um dicionário Python
pesos_lojas = {
    row["nome_cidade"]: row["peso_distribuicao"]
    for row in df_lojas.select("nome_cidade", "peso_distribuicao").collect()
}

print("Pesos originais:", pesos_lojas)

# Aplicando oscilação apenas nas lojas que NÃO são líderes fixas
# (identificamos a líder como a de maior peso-base do país)
loja_lider = max(pesos_lojas, key=pesos_lojas.get)

pesos_ajustados = {}
for cidade, peso in pesos_lojas.items():
    if cidade == loja_lider:
        pesos_ajustados[cidade] = peso  # peso da líder não oscila
    else:
        variacao = random.uniform(-0.20, 0.20)  # oscila até 20% para mais ou para menos
        pesos_ajustados[cidade] = peso * (1 + variacao)

# Normalizando para que a soma continue sendo 1.0 (100%)
soma_pesos = sum(pesos_ajustados.values())
pesos_finais = {cidade: peso / soma_pesos for cidade, peso in pesos_ajustados.items()}

print("Pesos finais (após oscilação):", pesos_finais)
print("Soma dos pesos finais:", sum(pesos_finais.values()))

In [0]:
quantidade_por_loja = {}
soma_alocada = 0

lojas_ordenadas = list(pesos_finais.keys())

for i, cidade in enumerate(lojas_ordenadas):
    if i == len(lojas_ordenadas) - 1:
        # Última loja recebe o restante, evitando erro de arredondamento
        quantidade_por_loja[cidade] = volume_do_dia - soma_alocada
    else:
        quantidade = int(volume_do_dia * pesos_finais[cidade])
        quantidade_por_loja[cidade] = quantidade
        soma_alocada += quantidade

print("Quantidade de vendas por loja:", quantidade_por_loja)
print("Soma total:", sum(quantidade_por_loja.values()))
print("Volume esperado:", volume_do_dia)

In [0]:
produtos_lista = df_produtos.select(
    "produto_id", "preco_brasil_brl", "preco_argentina_ars", "preco_mexico_mxn"
).collect()

representantes_lista = df_representantes.select(
    "representante_id", "cargo", "centro_vinculado"
).collect()

coluna_preco_por_pais = {
    "brasil": "preco_brasil_brl",
    "argentina": "preco_argentina_ars",
    "mexico": "preco_mexico_mxn",
}
moeda_por_pais = {"brasil": "BRL", "argentina": "ARS", "mexico": "MXN"}

coluna_preco = coluna_preco_por_pais[pais_selecionado]
moeda_atual = moeda_por_pais[pais_selecionado]

vendas_geradas = []

for cidade, quantidade_vendas_da_loja in quantidade_por_loja.items():

    representantes_da_loja = [
        r for r in representantes_lista
        if r["centro_vinculado"] == cidade or r["cargo"] == "Gerente"
    ]

    for _ in range(quantidade_vendas_da_loja):
        produto = random.choice(produtos_lista)
        representante = random.choice(representantes_da_loja)
        quantidade_itens = random.randint(1, 10)
        preco_unitario_texto = getattr(produto, coluna_preco)  # mantém como string, "sujo", igual veio da origem

        vendas_geradas.append((
            str(uuid.uuid4()),
            data_venda.isoformat(),
            produto["produto_id"],
            cidade,
            representante["representante_id"],
            pais_filtro,
            quantidade_itens,
            preco_unitario_texto,
            moeda_atual,
        ))

print(f"Total de vendas geradas: {len(vendas_geradas)}")
print("Exemplo de venda:", vendas_geradas[0])

In [0]:
# Transformação em dataframe spark

colunas_vendas = [
    "venda_id", "data_venda", "produto_id", "loja_cidade",
    "representante_id", "pais", "quantidade", "valor_unitario_moeda_local", "moeda"
]

df_vendas = spark.createDataFrame(vendas_geradas, colunas_vendas)

df_vendas.display()

In [0]:
# Gravar o resultado como JSON na Landing Zone

dbutils.widgets.dropdown("forcar_recriacao", "false", ["true", "false"])
forcar_recriacao = dbutils.widgets.get("forcar_recriacao") == "true"

caminho_landing_vendas = f"/Volumes/poc_latam_food/landing/blob_simulado/vendas/pais={pais_selecionado}/data={data_venda.isoformat()}"

arquivos_existentes = []
try:
    arquivos_existentes = dbutils.fs.ls(caminho_landing_vendas)
except Exception:
    pass

if len(arquivos_existentes) > 0 and not forcar_recriacao:
    print(f"Já existem vendas geradas para {pais_selecionado} em {data_venda.isoformat()}. "
          "Nenhuma ação foi tomada (use o widget 'forcar_recriacao' = true para sobrescrever).")
else:
    (
        df_vendas
        .write
        .mode("overwrite")
        .json(caminho_landing_vendas)
    )
    print(f"Vendas gravadas com sucesso em: {caminho_landing_vendas}")

In [0]:
spark.read.json(caminho_log_execucao).display()